# Notebook 07: evading our own detectors (test time)

**Class 03_03.** We now evaluate the detectors of classes 03_01 and 03_02 as an adversary would, to learn what
their alarms really depend on. Everything runs on our own models and the public KDD99 data, in feature space.

**Threat model** (no threat model, no security claim):

| | this notebook |
|:--|:--|
| goal | an attack row scored below the threshold (*evasion*); the attack must still be the same attack |
| capability | change only the **mutable** features: duration, bytes, connection counts (volume and timing) |
| fixed | protocol, service, flag, error rates, content features: changing them changes what the connection does |
| knowledge | (a) normal data only (*mimicry*); (b) the score of each row, one **query** each (*black box*) |
| stage | test time; the detector is fixed (poisoning is notebook 08) |

**Metrics.** *Evasion rate*: among the attacks the detector catches, the share that passes after the change.
*Queries*: how many scores the attacker asked for.

In [ ]:
import os
import sys
import time

os.environ.setdefault("KERAS_BACKEND", "jax")
import numpy as np
import polars as pl
from matplotlib import pyplot as plt
from scipy.stats import chi2

sys.path.insert(0, ".")
import plots_robust as pr

import anomlib as an
import deeplib as dl
import robustlib as rl

plt.rcParams["figure.dpi"] = 100
an.table_config()

## A. Example 1: evasion has a price

Point B of notebook 00 sends a lot ($x_1=1$) and receives little ($x_2=-1$). To pass a Mahalanobis detector at 1%
false alarms ($d_M^2\le\chi^2_{2,0.99}=9.21$) while still receiving $x_2=-1$, how much can a session send? Solve
$(x_1^2-1.8x_1x_2+x_2^2)/0.19\le 9.21$ for $x_1$: a quadratic in $x_1$.

In [ ]:
t = chi2.ppf(0.99, 2)
x2 = -1.0
# x1^2 - 1.8 x2 x1 + x2^2 - 0.19 t <= 0
a, b, c = 1.0, -1.8 * x2, x2**2 - 0.19 * t
x1_max = (-b + np.sqrt(b * b - 4 * a * c)) / (2 * a)
print(f"threshold d^2 = {t:.2f}; with x2 = -1 the session may send at most x1 = {x1_max:.2f} (B had 1.00)")
pr.evasion_price(float(t))  # slide plots: assets/figures/plots/03/

B can pass, but only by sending **less per session** ($x_1\le 0.35$ instead of 1 on the standardised log scale):
the detector does not make exfiltration impossible, it makes it **slower** and therefore longer and easier to catch
by other means. That is the honest meaning of "robust": it raises the attacker's cost.

## B. KDD99: the victims

In [ ]:
sp = an.kdd_split(seed=0)
Xtr, Xva, Xte, y = sp.X("train"), sp.X("val"), sp.X("test"), sp.y_test
names = sp.prep.names
mut = np.array([n in rl.MUTABLE for n in names])
print("mutable features:", [n for n in names if n in rl.MUTABLE])
lo, hi = Xtr.min(0), Xtr.max(0)  # the attacker stays inside the range seen in normal traffic
Xatt, cat = Xte[y == 1], sp.test["category"].to_numpy()[y == 1]
V = {d.name: rl.Victim.build(d, Xtr, Xva) for d in
     [an.Mahalanobis(), an.GMM(), an.KNNDist(), an.OCSVM(), an.IForest(), an.PCARecon(), dl.AE(), dl.DeepSVDD()]}  # fmt: skip

## C. Mimicry: borrow the volume and timing of a similar normal connection

For each caught attack, find the normal training row closest in the **fixed** features and copy its mutable
features (Wagner and Soto 2002 called this a mimicry attack). No query is needed.

In [ ]:
rows = []
for k, v in V.items():
    caught = v.alarm(Xatt)
    Xm = rl.mimicry(Xatt[caught], Xtr, mut)
    passed = ~v.alarm(Xm)
    rows.append({"detector": k, "caught": caught.mean(), "evasion (mutable only)": passed.mean(),
                 **{c: passed[cat[caught] == c].mean() if (cat[caught] == c).any() else None for c in an.CATEGORIES}})  # fmt: skip
mim = pl.DataFrame(rows)
print(mim)
pr.mimicry(mim.to_dicts())

**Reading.** Copying volume and timing evades few of the caught attacks (2% for Mahalanobis, 12% for Deep SVDD).
The detectors catch these attacks by what they **do** (SYN errors, rejected connections, failed logins, root
shells), not by how big or how long they are. Most evasions are r2l and u2r rows: those already look like normal
logins and sit near the threshold.

**Defender's lesson:** a detector is as robust as the features it relies on are hard to change. Features that the
attack *needs* (flags, error rates, privileged operations) are robust; volume and timing are cheap to fake.

## D. Black-box search with a query budget

The attacker now gets a score per query and changes one mutable feature at a time (a random step, kept only if the
score goes down), inside the normal range, until the row passes or the budget runs out. The Isolation Forest is left
out (a query costs 10 ms, the search would take minutes). The last rows lift the constraint: **every** feature is
mutable (the row is then no longer the attack; this only shows what the constraint is worth).

In [ ]:
rows = []
for k in ["Mahalanobis", "GMM (-log-likelihood)", "kNN distance", "One-class SVM", "PCA reconstruction", "autoencoder"]:
    v = V[k]
    caught = np.flatnonzero(v.alarm(Xatt))[:60]
    for budget in [50, 200, 1000]:
        t0 = time.time()
        res = [rl.random_search(v, Xatt[i], mut, lo, hi, budget=budget, seed=int(i)) for i in caught]
        ok = [r for r in res if r[1]]
        rows.append({"detector": k, "budget": budget, "evasion": len(ok) / len(res),
                     "queries (successes)": np.mean([r[2] for r in ok]) if ok else None,
                     "evaded categories": ",".join(sorted({cat[i] for i, r in zip(caught, res) if r[1]})),
                     "seconds": time.time() - t0})  # fmt: skip
for k in ["Mahalanobis", "PCA reconstruction", "autoencoder"]:
    v = V[k]
    caught = np.flatnonzero(v.alarm(Xatt))[:60]
    res = [rl.random_search(v, Xatt[i], np.ones_like(mut), lo, hi, budget=1000, seed=int(i)) for i in caught]
    ok = [r for r in res if r[1]]
    rows.append(
        {
            "detector": k + " (all features)",
            "budget": 1000,
            "evasion": len(ok) / len(res),
            "queries (successes)": np.mean([r[2] for r in ok]) if ok else None,
            "evaded categories": ",".join(sorted({cat[i] for i, r in zip(caught, res) if r[1]})),
            "seconds": None,
        }
    )
bb = pl.DataFrame(rows)
print(bb)
pr.blackbox(bb.to_dicts())

With the constraint, even 1,000 queries per row evade at most 5% (a few DoS rows); without it the same search
evades 15-30% of the rows (Mahalanobis 0.20, AE 0.15, PCA 0.30; a random one-feature-at-a-time search is a weak attacker:
a gradient attack on the AE would do better). The feasible region, not the attacker's cleverness, is the limit. Compare with the spam
module, where adding words was always allowed and evasion reached 100%.
